[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/contiloop/scp_stage1/blob/main/notebooks/train.ipynb)

# Stage 1: 경제 도메인 Continued Pretraining

Qwen3.5-4B (VLM) + LoRA CPT (한국어 경제 모노 데이터)

**파이프라인**: 환경 셋업 → 로그인 → GPU 확인 → 전처리 → 학습 → 평가 → Merge & Push

## 0. Repo 클론 & 의존성 설치

In [ ]:
# ── Repo 클론 (이미 클론했으면 스킵) ──
import os
REPO_URL = "https://github.com/contiloop/scp_stage1.git"
REPO_DIR = "/content/scp_stage1"  # Colab 기본 경로

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print(f"작업 디렉토리: {os.getcwd()}")

In [ ]:
!make setup

## 1. HuggingFace & W&B 로그인

Colab Secrets을 사용하면 토큰을 안전하게 관리할 수 있습니다.
- Colab 왼쪽 🔑 아이콘 → `HF_TOKEN`, `WANDB_API_KEY` 추가
- 또는 아래 셀에서 직접 입력

In [ ]:
import os

# ── 방법 1: Colab Secrets (권장) ──
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("✓ Colab Secrets에서 토큰 로드 완료")
except (ImportError, Exception):
    print("Colab Secrets 사용 불가 → 아래에서 수동 로그인하세요")

# ── HuggingFace 로그인 ──
from huggingface_hub import login
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
    print("✓ HF 로그인 완료")
else:
    login()  # 대화형 입력

# ── W&B 로그인 ──
import wandb
if os.environ.get("WANDB_API_KEY"):
    wandb.login(key=os.environ["WANDB_API_KEY"])
    print("✓ W&B 로그인 완료")
else:
    wandb.login()  # 대화형 입력

In [ ]:
# ── 데이터셋 접근 확인 ──
from datasets import load_dataset

ds = load_dataset("json",
    data_files="hf://datasets/alwaysgood/ko-news-split-512/hk.jsonl",
    split="train", streaming=True)
s = next(iter(ds))
print(f"✓ ko-news-split-512 접근 가능! 컬럼: {list(s.keys())}")
print(f"  샘플: {s['content'][:100]}...")

## 2. GPU 확인

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

    if torch.cuda.get_device_capability(0)[0] >= 8:
        print("bf16 supported (Ampere+)")
    else:
        print("bf16 not supported - use fp16")
else:
    print("No GPU!")

## 3. 데이터 전처리

HF Hub에서 한국어+영어 데이터를 다운로드하여 토큰화/패킹합니다.
- 한국어: `alwaysgood/ko-news-split-512` (hk, mk, korea_bank, naver)
- 영어: `alwaysgood/earnings_call_mono` (135K earnings call chunks)

결과: `data/processed/train`, `data/processed/val` (Arrow 포맷)

In [ ]:
!make preprocess

In [ ]:
from datasets import load_from_disk
from transformers import AutoTokenizer

train_ds = load_from_disk("data/processed/train")
val_ds = load_from_disk("data/processed/val")
print(f"Train: {len(train_ds):,} seqs | Val: {len(val_ds):,} seqs")
print(f"Total tokens: {len(train_ds) * 4096:,}")

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3.5-4B-Base", trust_remote_code=True)
print(f"\n--- sample (first 200 tokens) ---")
print(tokenizer.decode(train_ds[0]["input_ids"][:200]))

## 4. Train

Config: `configs/stage1.yaml`

In [ ]:
!make train

## 5. 평가

Base 모델 대비 perplexity 개선 + 도메인 용어 완성 테스트

In [ ]:
!make eval BASE_MODEL=unsloth/Qwen3.5-4B

## 6. Merge & Push

In [ ]:
HF_REPO = "alwaysgood/qwen3.5-4b-econ-ko"

# merge + push
!make push HF_REPO={HF_REPO}

In [ ]:
# adapter only (lighter)
# !make push-adapter HF_REPO={HF_REPO}

## 완료!

모델이 HF Hub에 업로드되었습니다. 다음 단계:
- **Stage 2**: 번역 parallel data로 SFT → `stage2_sft/`
- **추론 테스트**:
```python
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("alwaysgood/qwen3.5-4b-econ-ko")
```